In [1]:
import pandas as pd, re, math
from collections import defaultdict

data = pd.read_csv("spam.csv", encoding="latin1")
data = data[['v1', 'v2']].rename(columns={'v1':'label','v2':'text'})


In [2]:
key_words = ["rich","buy","friend","coffee","won","free","report","meeting","business"]

In [3]:
def count_words(text):
    text = str(text).lower()
    words = re.findall(r"[a-z]+", text)
    return [words.count(k) for k in key_words]

occurrence = pd.DataFrame(data['text'].apply(count_words).tolist(), columns=key_words)
occurrence['label'] = data['label']
occurrence.head()


,rich,buy,friend,coffee,won,free,report,meeting,business,label
0,0,0,0,0,0,0,0,0,0,ham
1,0,0,0,0,0,0,0,0,0,ham
2,0,0,0,0,0,1,0,0,0,spam
3,0,0,0,0,0,0,0,0,0,ham
4,0,0,0,0,0,0,0,0,0,ham


In [4]:
total = len(occurrence)
spam_count = sum(occurrence['label']=="spam")
ham_count = sum(occurrence['label']=="ham")

P_spam = spam_count / total
P_ham  = ham_count / total


In [5]:
spam_words = occurrence[occurrence['label']=="spam"][key_words]
ham_words  = occurrence[occurrence['label']=="ham"][key_words]

likelihood_spam = (spam_words.sum()+1) / (len(spam_words)+2)
likelihood_ham  = (ham_words.sum()+1) / (len(ham_words)+2)


In [6]:
model = pd.DataFrame({
    "word": key_words,
    "P(word|Spam)": likelihood_spam.values,
    "P(word|Ham)": likelihood_ham.values
})
model["P(Spam)"] = P_spam
model["P(Ham)"]  = P_ham
print(model)


       word  P(word|Spam)  P(word|Ham)   P(Spam)    P(Ham)
0      rich      0.001335     0.000829  0.134063  0.865937
1       buy      0.008011     0.013052  0.134063  0.865937
2    friend      0.016021     0.008701  0.134063  0.865937
3    coffee      0.001335     0.001865  0.134063  0.865937
4       won      0.102804     0.004143  0.134063  0.865937
5      free      0.305741     0.012637  0.134063  0.865937
6    report      0.001335     0.001450  0.134063  0.865937
7   meeting      0.001335     0.009115  0.134063  0.865937
8  business      0.001335     0.000829  0.134063  0.865937


In [9]:
def classify(email):
    text = re.findall(r"[a-z]+", email.lower())
    p_spam = math.log(P_spam)
    p_ham  = math.log(P_ham)
    for word in key_words:
        if word in text:
            p_spam += math.log(likelihood_spam[word])
            p_ham  += math.log(likelihood_ham[word])
        else:
            p_spam += math.log(1 - likelihood_spam[word])
            p_ham  += math.log(1 - likelihood_ham[word])
    return "spam" if p_spam > p_ham else "ham"

test_email = "Win a free iPhone and get rich now!"
print("Prediction:", classify(test_email))


Prediction: spam
